# Spectral Waterfall Minimal

Compact notebook front end for the production spectral-waterfall workflow.

Open it with the same Python environment as `run_spectral_waterfall.py`.

Use it when you want to:
- point the workflow at a different config or run
- narrow the figure to selected experiments, stations, or size ranges
- explore the spectral waterfall **interactively** with a time (`itime`) slider (no PNG export)

Optional batch export (PNG frames + MP4) still lives on `SpectralWaterfallNotebook.render` if you call it from code; this notebook focuses on the slider workflow.

**Footer ice ⟨D⟩:** very small ridge-integrated ice can inflate the mean diameter. `growth_ice_sum_floor_q` and `growth_ice_mask_until_min` in `psd_process_evolution.yaml` hide those early-time artifacts.

In [ ]:
from __future__ import annotations

import argparse
import csv
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repo root so imports and output paths stay stable from any notebook cwd."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "src/utilities/spectral_waterfall.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook inside the polarcap_analysis repository.")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
for path in (REPO_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from utilities.process_budget_data import load_process_budget_data
from utilities.process_rates import first_plume_ridge_anchor
from utilities.style_profiles import apply_publication_style
from utilities.spectral_waterfall import (
    _MinMaxAction,
    _build_mp4,
    _build_time_window,
    _ffmpeg_path,
    _group_existing_frames,
    _precompute_growth_overlays,
    _read_yaml,
    _remove_existing_frames,
    _save_frame,
    _sorted_frame_paths,
    _station_tag,
    _waterfall_cfg,
    plot_spectral_waterfall,
)
from scripts.processing_chain.run_publication_figures import _load_publication_config

apply_publication_style()
print(f"repo: {REPO_ROOT}")

In [ ]:
@dataclass
class SpectralWaterfallSettings:
    '''Small notebook control surface; keep stable defaults in YAML.'''

    kind: str = 'Q'  # Choose the displayed unit family: mass (`Q`) or number (`N`).
    config_path: Path = REPO_ROOT / 'config/psd_process_evolution.yaml'  # PSD + process-evolution defaults (shared process-budget YAML).
    publication_config_path: Path = REPO_ROOT / 'config/publication_figures.yaml'  # Reuse publication presets.
    use_publication_defaults: bool = False  # Start from paper-style limits and MP4 defaults.
    exp_ids: list[int] | None = None  # Restrict the run to selected experiment ids.
    range_keys: list[str] | None = None  # Restrict the figure to selected size families.
    station_ids: list[int] | None = None  # Restrict the figure to selected station columns.
    workers: int | None = None  # Trade render speed against memory use and debugging simplicity.
    sw_overrides: dict[str, Any] = field(default_factory=dict)  # Override a few waterfall options without editing YAML.


class SpectralWaterfallNotebook:
    '''Notebook runner that keeps the production render path but trims the interface to a few high-impact switches.'''

    def __init__(self, settings: SpectralWaterfallSettings):
        self.settings = settings
        self.cfg_yaml = _read_yaml(settings.config_path)
        self.cfg = load_process_budget_data(REPO_ROOT, config_path=settings.config_path)
        self.sw_cfg = _waterfall_cfg(self.cfg_yaml, kind_hint=settings.kind.upper())
        self.sw_cfg['kind'] = settings.kind.upper()
        self.default_mp4 = False
        if settings.use_publication_defaults:
            self.default_mp4 = self._apply_publication_defaults()
        self.sw_cfg.update(settings.sw_overrides)

        self.kind = self.sw_cfg['kind']
        self.kind_label = 'number' if self.kind == 'N' else 'mass'
        self.kind_dir = 'N' if self.kind == 'N' else 'M'
        self.cs_run = self.cfg_yaml.get('ensemble', {}).get('cs_run', 'unknown_cs_run')
        self.cs_run_tag = self.cs_run.replace('/', '_')

        selection = self.cfg_yaml.get('selection', {})
        plotting = self.cfg_yaml.get('plotting', {})
        render = plotting.get('render', {})

        self.station_ids = settings.station_ids or selection.get('plot_station_ids', self.cfg['plot_stn_ids'])
        self.plot_exp_ids = settings.exp_ids or selection.get('plot_experiment_ids', self.cfg['plot_exp_ids'])
        self.plot_range_keys = settings.range_keys or plotting.get('plot_range_keys', self.cfg['plot_range_keys'])
        self.height_sel_m = plotting.get('height_sel_m', self.cfg['height_sel_m'])
        self.time_window = _build_time_window(self.cfg_yaml, self.cfg)

        self.frame_root = REPO_ROOT / 'output/gfx/png/05' / self.cs_run
        self.mp4_root = REPO_ROOT / 'output/gfx/mp4'
        self.csv_root = REPO_ROOT / 'output/gfx/csv/05' / self.cs_run_tag
        self.frame_dpi = int(render.get('frame_dpi', 300))
        self.frame_png_compress = int(render.get('frame_png_compress', 1))
        self.mp4_fps = int(render.get('mp4_fps', 5))
        default_workers = settings.workers if settings.workers is not None else render.get('workers', 4)
        self.workers = max(1, int(default_workers))
        self.stn_tag = _station_tag(self.station_ids)
        self.ffmpeg_cmd = _ffmpeg_path()

        self.ridge_context = self._build_ridge_context()
        (
            self.growth_overlays,
            self.growth_csv_rows,
            self.growth_series,
            self.ridge_ref_height_m,
        ) = _precompute_growth_overlays(
            plot_exp_ids=self.plot_exp_ids,
            plot_range_keys=self.plot_range_keys,
            station_ids=self.station_ids,
            time_window=self.time_window,
            cfg=self.cfg,
            kind=self.kind,
            sw_cfg=self.sw_cfg,
            ridge_context=self.ridge_context,
            ds=self.cfg.get("ds"),
        )
        self.growth_csv_path = self._write_growth_csv()

    def _apply_publication_defaults(self) -> bool:
        publication_cfg = _load_publication_config(self.settings.publication_config_path)
        lookup_key = f'spectral_waterfall_{self.sw_cfg["kind"]}'
        tokens = publication_cfg.get(lookup_key, [])
        if not tokens:
            return False

        parser = argparse.ArgumentParser(add_help=False)
        parser.add_argument('--kind', choices=['N', 'Q', 'n', 'q'], default=None)
        parser.add_argument('--normalize-mode', choices=['none', 'bin', 'panel'], default=None)
        parser.add_argument('--plot-style', choices=['bars', 'lines', 'steps'], default=None)
        parser.add_argument('--yscale', choices=['symlog', 'linear', 'log'], default=None)
        parser.add_argument('--linthresh-w', type=float, default=None)
        parser.add_argument('--linthresh-f', type=float, default=None)
        parser.add_argument('--linscale', type=float, default=None)
        parser.add_argument('--xlim-w', action=_MinMaxAction, default=None)
        parser.add_argument('--xlim-f', action=_MinMaxAction, default=None)
        parser.add_argument('--ylim-w', action=_MinMaxAction, default=None)
        parser.add_argument('--ylim-f', action=_MinMaxAction, default=None)
        parser.add_argument('--psd-ylim-w', action=_MinMaxAction, default=None)
        parser.add_argument('--psd-ylim-f', action=_MinMaxAction, default=None)
        parser.add_argument('--workers', type=int, default=None)
        parser.add_argument('--mp4', action='store_true')
        parser.add_argument('--mp4-only', action='store_true')
        parser.add_argument('--no-psd-twin', action='store_true')
        ns = parser.parse_args(tokens)

        if ns.kind is not None:
            self.sw_cfg['kind'] = ns.kind.upper()

        scalar_map = {
            'normalize_mode': 'normalize_mode',
            'plot_style': 'plot_style',
            'yscale': 'yscale',
            'linthresh_w': 'linthresh_W',
            'linthresh_f': 'linthresh_F',
            'linscale': 'linscale',
        }
        range_map = {
            'xlim_w': 'xlim_W',
            'xlim_f': 'xlim_F',
            'ylim_w': 'ylim_W',
            'ylim_f': 'ylim_F',
            'psd_ylim_w': 'psd_ylim_W',
            'psd_ylim_f': 'psd_ylim_F',
        }
        for attr, key in scalar_map.items():
            value = getattr(ns, attr)
            if value is not None:
                self.sw_cfg[key] = value
        for attr, key in range_map.items():
            value = getattr(ns, attr)
            if value is not None:
                self.sw_cfg[key] = value[0]
        if ns.no_psd_twin:
            self.sw_cfg['show_psd_twin'] = False
        if ns.workers is not None and self.settings.workers is None:
            self.settings.workers = ns.workers
        return bool(ns.mp4 or ns.mp4_only)

    def _build_ridge_context(self) -> dict[tuple[int, str], dict[int, tuple[int, int, float]]]:
        floor = float(self.sw_cfg.get('ridge_mass_floor', 0.0))
        out: dict[tuple[int, str], dict[int, tuple[int, int, float]]] = {}
        for exp_id in self.plot_exp_ids:
            rates = self.cfg['rates_by_exp'][exp_id]
            ridge_conc = rates.get('spec_conc_Q_F')
            if ridge_conc is None:
                continue
            for range_key in self.plot_range_keys:
                bin_slice = self.cfg['size_ranges'][range_key]['slice']
                per_station: dict[int, tuple[int, int, float]] = {}
                for station_id in self.station_ids:
                    anchor = first_plume_ridge_anchor(ridge_conc, station_id, bin_slice, floor=floor)
                    if anchor is None:
                        continue
                    gti, hi = anchor
                    z_m = float(np.asarray(ridge_conc['height_level'].values)[int(hi)])
                    per_station[station_id] = (gti, hi, z_m)
                out[(exp_id, range_key)] = per_station
        return out

    def _write_growth_csv(self) -> Path | None:
        if not (self.sw_cfg.get('write_growth_csv', False) and self.growth_csv_rows):
            return None
        self.csv_root.mkdir(parents=True, exist_ok=True)
        csv_path = self.csv_root / f'ridge_growth_{self.kind}_{self.stn_tag}.csv'
        with csv_path.open('w', newline='', encoding='utf-8') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(self.growth_csv_rows[0].keys()))
            writer.writeheader()
            writer.writerows(self.growth_csv_rows)
        return csv_path

    def summary(self) -> dict[str, Any]:
        return {
            'cs_run': self.cs_run,
            'kind': self.kind,
            'stations': self.station_ids,
            'experiments': self.plot_exp_ids,
            'ranges': self.plot_range_keys,
            'n_frames': len(self.time_window) - 1,
            'frame_root': str(self.frame_root),
            'mp4_root': str(self.mp4_root),
            'growth_csv': None if self.growth_csv_path is None else str(self.growth_csv_path),
        }

    def interactive_time(
        self,
        exp_id: int | None = None,
        range_key: str | None = None,
        *,
        continuous_slider: bool = False,
    ) -> None:
        '''Redraw the waterfall inside the notebook from an `itime` slider; does not write PNGs.'''
        exp_id = self.plot_exp_ids[0] if exp_id is None else exp_id
        range_key = self.plot_range_keys[0] if range_key is None else range_key
        n = len(self.time_window) - 1
        if n <= 0:
            raise ValueError('time_window yields no frames')

        out = widgets.Output(layout={'border': '1px solid #ddd'})
        slider = widgets.IntSlider(
            value=0, min=0, max=n - 1, step=1, continuous_update=continuous_slider, description='itime',
        )
        _last_fig: list[Any | None] = [None]

        def redraw(itime: int) -> None:
            with out:
                out.clear_output(wait=True)
                if _last_fig[0] is not None:
                    plt.close(_last_fig[0])
                _last_fig[0] = self.preview(itime=int(itime), exp_id=exp_id, range_key=range_key, save=False)
                plt.show()

        def on_change(change: dict[str, Any]) -> None:
            redraw(int(change['new']))

        slider.observe(on_change, names='value')
        display(widgets.VBox([slider, out]))
        redraw(0)

    def _render_figure(self, exp_id: int, range_key: str, itime: int):
        rates = self.cfg['rates_by_exp'][exp_id]
        twindow = slice(self.time_window[itime], self.time_window[itime + 1])

        ctx = self.ridge_context.get((exp_id, range_key), {})
        anchor_map = {station: (gti, hi) for station, (gti, hi, _z) in ctx.items()}
        ref_height = self.ridge_ref_height_m.get((exp_id, range_key), {})
        fallback_height = {station: z_m for station, (_gti, _hi, z_m) in ctx.items()}
        ridge_height = {
            station: (
                float(ref_height[station])
                if station in ref_height and np.isfinite(ref_height[station])
                else fallback_height[station]
            )
            for station in self.station_ids
            if (station in ref_height and np.isfinite(ref_height[station])) or station in fallback_height
        }

        show_psd = bool(self.sw_cfg.get('show_psd_twin'))
        growth_footer = bool(self.sw_cfg.get('show_growth_footer') or self.sw_cfg.get('show_growth_sparkline'))
        fig, _ = plot_spectral_waterfall(
            spec_rates_w=rates[f'spec_rates_{self.kind}_W'],
            spec_rates_f=rates[f'spec_rates_{self.kind}_F'],
            size_ranges=self.cfg['size_ranges'],
            range_key=range_key,
            diameter_um=self.cfg['diameter_um'],
            station_ids=self.station_ids,
            station_labels=self.cfg['station_labels'],
            height_sel_m=self.height_sel_m,
            twindow=twindow,
            unit_label=rates[f'unit_{self.kind}'],
            kind_label=self.kind_label,
            cfg_plot=self.sw_cfg,
            normalize_mode=self.sw_cfg['normalize_mode'],
            plot_style=self.sw_cfg['plot_style'],
            yscale=self.sw_cfg['yscale'],
            spec_conc_w=rates.get(f'spec_conc_{self.kind}_W') if show_psd else None,
            spec_conc_f=rates.get(f'spec_conc_{self.kind}_F') if show_psd else None,
            ridge_conc_f=rates.get('spec_conc_Q_F'),
            conc_unit_label=r'#/m³' if self.kind == 'N' else r'g/m³',
            ridge_anchor_by_station=anchor_map or None,
            ridge_height_m_by_station=ridge_height or None,
            growth_overlay_by_station=self.growth_overlays.get((exp_id, range_key, itime)),
            growth_footer_series_by_station=self.growth_series.get((exp_id, range_key)) if growth_footer else None,
            growth_footer_itime=itime,
            growth_footer_slice_unit='g/m3' if self.kind == 'Q' else '#/m3',
        )
        stem = f'spectral_waterfall_{self.kind}_{self.cs_run_tag}_exp{exp_id}_{self.stn_tag}_{range_key}_itime{itime}'
        out_dir = self.frame_root / f'exp{exp_id}' / self.kind_dir
        return fig, stem, out_dir

    def preview(self, itime: int = 0, exp_id: int | None = None, range_key: str | None = None, save: bool = False):
        exp_id = self.plot_exp_ids[0] if exp_id is None else exp_id
        range_key = self.plot_range_keys[0] if range_key is None else range_key
        fig, stem, out_dir = self._render_figure(exp_id, range_key, itime)
        if save:
            _save_frame(fig, stem, out_dir, self.frame_dpi, self.frame_png_compress)
        return fig

    def _save_render_task(self, task: tuple[int, str, int]) -> str:
        exp_id, range_key, itime = task
        fig, stem, out_dir = self._render_figure(exp_id, range_key, itime)
        _save_frame(fig, stem, out_dir, self.frame_dpi, self.frame_png_compress)
        plt.close(fig)
        return stem

    def build_mp4_only(self) -> list[Path]:
        groups = _group_existing_frames(
            self.frame_root,
            kind=self.kind,
            kind_dir=self.kind_dir,
            cs_run_tag=self.cs_run_tag,
            exp_ids=self.plot_exp_ids,
            range_keys=self.plot_range_keys,
            station_tag=self.stn_tag,
        )
        mp4_paths: list[Path] = []
        for prefix, frames in groups:
            mp4_path = self.mp4_root / f'{prefix}_evolution_nframes{len(frames)}.mp4'
            _build_mp4(self.ffmpeg_cmd, frames, mp4_path, self.mp4_fps)
            mp4_paths.append(mp4_path)
            print(mp4_path)
        return mp4_paths

    def render(self, *, mp4: bool | None = None, mp4_only: bool = False) -> list[Path]:
        mp4 = self.default_mp4 if mp4 is None else mp4
        self.frame_root.mkdir(parents=True, exist_ok=True)
        self.mp4_root.mkdir(parents=True, exist_ok=True)
        if mp4_only:
            return self.build_mp4_only()

        started = time.perf_counter()
        expected_frames = len(self.time_window) - 1
        mp4_paths: list[Path] = []

        for exp_id in self.plot_exp_ids:
            for range_key in self.plot_range_keys:
                frame_dir = self.frame_root / f'exp{exp_id}' / self.kind_dir
                pattern = str(
                    frame_dir
                    / f'spectral_waterfall_{self.kind}_{self.cs_run_tag}_exp{exp_id}_{self.stn_tag}_{range_key}_itime*.png'
                )
                _remove_existing_frames(pattern)
                tasks = [(exp_id, range_key, itime) for itime in range(expected_frames)]

                if self.workers > 1 and len(tasks) > 1:
                    with ThreadPoolExecutor(max_workers=min(self.workers, len(tasks))) as pool:
                        list(pool.map(self._save_render_task, tasks))
                else:
                    for task in tasks:
                        self._save_render_task(task)

                frames = _sorted_frame_paths(pattern, expected_count=expected_frames)
                print(f'exp={exp_id} {range_key}: {len(frames)} frames')
                if mp4 and frames:
                    mp4_path = self.mp4_root / (
                        f'spectral_waterfall_{self.kind}_{self.cs_run_tag}_exp{exp_id}_{self.stn_tag}_{range_key}_evolution_nframes{len(frames)}.mp4'
                    )
                    _build_mp4(self.ffmpeg_cmd, frames, mp4_path, self.mp4_fps)
                    mp4_paths.append(mp4_path)
                    print(mp4_path)

        print(f'done in {time.perf_counter() - started:.1f}s')
        return mp4_paths

    def show_mp4(self, path: Path) -> None:
        display(Video(str(path), embed=True, html_attributes='controls loop'))

In [ ]:
SETTINGS = SpectralWaterfallSettings(
    kind='Q',  # Start in mass mode.
    use_publication_defaults=True,  # Pull in the paper-style limits and MP4 preset.
    exp_ids=[0],  # Focus on one experiment.
    range_keys=['ALLBB'],  # Use the full diameter family.
    station_ids=[0,1,2],  # Show seeding, observation, and precipitation sites together.
    workers=4,  # Speed up a full render; lower this if memory gets tight.
    sw_overrides={
        'show_psd_twin': True,  # Keep PSD context above each station column.
        'show_growth_footer': True,  # Keep the four-row growth footer visible.
    },
)

runner = SpectralWaterfallNotebook(SETTINGS)
runner.summary()

In [ ]:
# Interactive time: slider updates the figure; no PNG or MP4 files are written.
# Set continuous_slider=True to redraw while dragging (heavier).
runner.interactive_time()


In [ ]:
# Optional batch export (writes PNG frames under output/gfx/png/... and may build MP4):
# mp4_paths = runner.render(mp4=True)
# if mp4_paths:
#     runner.show_mp4(mp4_paths[0])
